# So lets start

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\rusha\AppData\Local\Temp\ipykernel_56220\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents



In [4]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("sources/")

Found 1 PDF files to process

Processing: RushalMananadhar(790333).pdf
  ✓ Loaded 15 pages

Total documents loaded: 15


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='0  \nAssignment  II  \nDistributed  and  Cloud  Computing  \n \nChapter  9  \nEmerging  Trends  in  Distributed  and  Cloud  Computing   \n \n \n \n \n \n \n \n \n \n \nRushal  Manandhar  \n790333  \nBE.  Computer  \nVII  semester  \n \nDate  :  9  August,  2026'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 1, 'page_label': '2', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='Page  |  1  \nTable  Of  Content  \nEdge  Computing  and  Fog  Computing ...................

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 15 documents into 37 chunks

Example chunk:
Content: 0  
Assignment  II  
Distributed  and  Cloud  Computing  
 
Chapter  9  
Emerging  Trends  in  Distributed  and  Cloud  Computing   
 
 
 
 
 
 
 
 
 
 
Rushal  Manandhar  
790333  
BE.  Computer  
VI...
Metadata: {'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='0  \nAssignment  II  \nDistributed  and  Cloud  Computing  \n \nChapter  9  \nEmerging  Trends  in  Distributed  and  Cloud  Computing   \n \n \n \n \n \n \n \n \n \n \nRushal  Manandhar  \n790333  \nBE.  Computer  \nVII  semester  \n \nDate  :  9  August,  2026'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 1, 'page_label': '2', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='Page  |  1  \nTable  Of  Content'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Do

### Embedding And vectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10000.77it/s]


Model loaded successfully. Embedding dimension: 384


In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "datas/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 111


In [11]:
chunks

[Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='0  \nAssignment  II  \nDistributed  and  Cloud  Computing  \n \nChapter  9  \nEmerging  Trends  in  Distributed  and  Cloud  Computing   \n \n \n \n \n \n \n \n \n \n \nRushal  Manandhar  \n790333  \nBE.  Computer  \nVII  semester  \n \nDate  :  9  August,  2026'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Distributed Solo', 'source': 'sources\\RushalMananadhar(790333).pdf', 'total_pages': 15, 'page': 1, 'page_label': '2', 'source_file': 'RushalMananadhar(790333).pdf', 'file_type': 'pdf'}, page_content='Page  |  1  \nTable  Of  Content'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Do

In [12]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 37 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.15it/s]

Generated embeddings with shape: (37, 384)
Adding 37 documents to vector store...
Successfully added 37 documents to vector store
Total documents in collection: 148


In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("what is edge computing")

Retrieving documents for query: 'what is edge computing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 119.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8c83c69c_7',
  'content': 'Page  |  2  \nEdge  Computing  and  Fog  Computing  \n1.1  Why  they  exist  \nTraditional  cloud  computing  sends  all  data  from  devices  to  a  centralized  data  center  for  \nprocessing.\n \nAs\n \nIoT\n \ndevices,\n \nsensors,\n \nand\n \nsmart\n \nsystems\n \ngrew\n \nmassively\n \nin\n \nnumber,\n \nthis\n \ncreated\n \nproblems:\n \nhigh\n \nlatency,\n \nheavy\n \nbandwidth\n \nuse,\n \nand\n \npoor\n \nperformance\n \nfor\n \ntime-sensitive\n \napplications\n \n(e.g.,\n \nautonomous\n \nvehicles,\n \nindustrial\n \nautomation).\n \nEdge\n \nand\n \nfog\n \ncomputing\n \nsolve\n \nthis\n \nby\n \nmoving\n \ncomputation\n \ncloser\n \nto\n \nwhere\n \ndata\n \nis\n \ngenerated\n.\n \n \n1.2  Edge  Computing  \nDefinition:  Processing  data  directly  on  or  very  near  the  device  that  generates  it  (sensors,  \nsmartphones,\n \nIoT\n \ngateways,\n \ncameras)\n \n—\n \nat\n \nthe\n \n"edge"\n \nof\n \nthe\n \nnetwork,\n \ninstead\

### RAG Pipeline- VectorDB To LLM Output Generation

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()
# print(os.getenv("GROQ_API_KEY"))

True

In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [18]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [19]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [20]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("cloud computing")

Retrieving documents for query: 'cloud computing'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 150.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_da6b3ab7_34',
  'content': 'Page  |  14  \nConclusion  \nThese  four  topics  are  deeply  interconnected  pieces  of  the  modern  distributed  and  cloud  \ncomputing\n \nlandscape.\n \nEdge/fog\n \ncomputing\n \naddresses\n \nwhere\n \ndata\n \nis\n \nprocessed,\n \nserverless\n \narchitecture\n \naddresses\n \nhow\n \ncode\n \nexecution\n \nand\n \ninfrastructure\n \nare\n \nmanaged,\n \nand\n \nDocker/Kubernetes\n \ntogether\n \nwith\n \ncloud-native/microservices\n \naddress\n \nhow\n \napplications\n \nare\n \nbuilt,\n \npackaged,\n \nand\n \nscaled\n \nreliably.\n \nCollectively,\n \nthey\n \nrepresent\n \nthe\n \nshift\n \nfrom\n \ncentralized,\n \nmonolithic,\n \nmanually-managed\n \nsystems\n \ntoward\n \ndistributed,\n \nautomated,\n \nand\n \nhighly\n \nscalable\n \ncomputing\n \n—\n \nwhich\n \nnow\n \ndefines\n \nthe\n \nmajority\n \nof\n \nmodern\n \nenterprise\n \nIT\n \ninfrastructure.\n \n \n \nReference   \n1.   Bonomi,  F.,  et  al.  (2012).  "Fog  com

In [21]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [22]:
answer=rag_simple("what is edge computing",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is edge computing'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 222.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Edge computing is the processing of data directly on or very near the device that generates it (sensors, smartphones, IoT gateways, cameras) — at the "edge" of the network, instead of sending it to a distant cloud server.


### Using local ollama for answering from the rag 3B instead of 11B

In [23]:
# import ollama

# def rag_simple2(query,retriever,top_k=3):
#     ## retriever the context
#     results=retriever.retrieve(query,top_k=top_k)
#     context="\n\n".join([doc['content'] for doc in results]) if results else ""
#     if not context:
#         return "No relevant context found to answer the question."
    
#     ## generate the answwer using GROQ LLM
#     response = ollama.chat(model='llama3.2', messages=[
#         {"role": "system", "content": f"Answer using this context:\n{context} in simply 3 4 lines"},
#         {"role": "user", "content": query}
#     ])

#     return response['message']['content']

In [24]:
# answer=rag_simple2("what is edge computing",rag_retriever)
# print(answer)

### Enhanced RAG Pipeline Features

In [25]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [26]:
# Example usage:
result = rag_advanced("Edge Computing", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Edge Computing'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 241.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Edge Computing is a method of processing data directly on or very near the device that generates it, at the "edge" of the network, instead of sending it to a distant cloud server.
Sources: [{'source': 'RushalMananadhar(790333).pdf', 'page': 2, 'score': 0.4102960228919983, 'preview': 'Page  |  2  \nEdge  Computing  and  Fog  Computing  \n1.1  Why  they  exist  \nTraditional  cloud  computing  sends  all  data  from  devices  to  a  centralized  data  center  for  \nprocessing.\n \nAs\n \nIoT\n \ndevices,\n \nsensors,\n \nand\n \nsmart\n \nsystems\n \ngrew\n \nmassively\n \nin\n \nnumber,\n \nthis\n \ncreated\n...'}, {'source': 'RushalMananadhar(790333).pdf', 'page': 2, 'score': 0.4102960228919983, 'preview': 'Page  |  2  \nEdge  Computing  and  Fog  Computing  \n1.1  Why  they  exist  \nTraditional  cloud  computing  sends  all  data  from  devices  to  a  centralized  data  center  for  \nprocessing.\n \nAs\n \nIoT\n \ndevices,\n \nsensors,\n \nand\n \nsmart\n \nsystems\n \ngre

In [27]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Docker and Kubernetes", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


Retrieving documents for query: 'Docker and Kubernetes'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 183.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Page  |  8  
●  Docker  Hub:  A  public  registry  to  store  and  share  container  images.  
Why  containers  instead  of  VMs:  Virtual  machines  virtualize  entire  hardware  and  need  a  full  
guest
 
OS
 
each,
 
making
 
them
 
heavy
 
(GBs,


 
minutes
 
to
 
boot).
 
Containers
 
virtualize
 
only
 
at
 
the
 
OS
 
level,
 
so
 
they're
 
lightweight
 
(MBs,
 
boot
 
in
 
seconds)
 
and
 
far
 
more
 
efficient
 
for
 
running
 
many
 
isolated
 
apps
 
on
 
one
 
machine.
 
3.2  Kubernetes  (K8s)  —  Container  Orchestration  
Definition:  Kubernetes  is  an  open-source  system  (originally  built  by  Google)  for  automating  
the
 
deployment,
 
scaling,
 
networking,
 
and
 
management
 
of
 
containers
 
across
 
a
 
cluster
 
of
 
machines.
 
Once
 
an
 
application
 
is
 
split
 
into
 
many
 
containers
 
(common
 
in
 
microservices),
 
manually
 
managing
 
them
 
becomes
 
impossible
 
at
 
scale
 
—
 
Kubernetes
 
automates
 
this.
 
Core  functions:

Page  |  8  
●  Docker  Hub:  A  public  registry  to  store  and  share  container  images.  
Why  containers  instead  of  VMs:  Virtual  machines  virtualize  entire  hardware  and  need  a  full  
guest
 
OS
 
each,
 
making
 
them
 
heavy
 
(GBs,
 
minutes

In [28]:
from SaveIt import saveMD

saveMD("hiiii")

In [45]:
from SaveIt import *

saves = []
saves.append(result['answer'])
saves.append(result['summary'])
saves.append(result['history'][-1])
# try:
#     
# Exception

try:
    for save in saves:
        if isinstance(save,Dict):
            saveJSON(save)
        else:
            saveMD(save)

except Exception as er:  # ← 'as er' captures the exception
    print("Error:", er)  # ← Now 'er' is defined